### Library

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.layers import Flatten,Dense
import keras_tuner as kt
from model import build_model,build_cnn_model

In [2]:
(train_ds, val_ds, test_ds), ds_info = tfds.load(
    'mnist', 
    split=['train[:80%]', 'train[80%:]', 'test'], 
    as_supervised=True,
    with_info=True
)

### Dataset pipeline

In [3]:
def normalize_img(ds, y):
    return tf.cast(ds, tf.float32) / 255., y

BATCH_SIZE=128

train_ds = train_ds.map(normalize_img, num_parallel_calls=tf.data.AUTOTUNE).cache().shuffle(len(train_ds)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(normalize_img, num_parallel_calls=tf.data.AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(normalize_img, num_parallel_calls=tf.data.AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


### Train

In [3]:
# build_cnn_model, or build_model depends on your config

tuner = kt.Hyperband(hypermodel=build_model, objective='val_accuracy',  max_epochs=10,factor=3,directory='logs',project_name='mnist_tuning')
# tuner = kt.Hyperband(hypermodel=build_cnn_model, objective='val_accuracy',  max_epochs=10,factor=3,directory='logs',project_name='mnist_tuning')


Reloading Tuner from logs/mnist_tuning/tuner0.json


In [6]:
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

In [ ]:
tuner.search(train_ds, epochs=3, validation_data=val_ds, callbacks=[stop_early])

Trial 30 Complete [00h 02m 23s]
val_accuracy: 0.9732499718666077

Best val_accuracy So Far: 0.9797499775886536
Total elapsed time: 00h 28m 14s


In [11]:
tuner.results_summary(num_trials=1)

# 2. Extract the absolute best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"Optimal Learning Rate: {best_hps.get('learning_rate')}")
print(f"Optimal Weight Decay: {best_hps.get('weight_decay')}")

Results summary
Results in logs/mnist_tuning
Showing 1 best trials
Objective(name="val_accuracy", direction="max")

Trial 0024 summary
Hyperparameters:
learning_rate: 0.0023318041518488285
weight_decay: 0.0009885379679405871
tuner/epochs: 10
tuner/initial_epoch: 4
tuner/bracket: 1
tuner/round: 1
tuner/trial_id: 0021
Score: 0.9797499775886536
Optimal Learning Rate: 0.0023318041518488285
Optimal Weight Decay: 0.0009885379679405871


In [12]:
model = tuner.hypermodel.build(best_hps)

In [13]:
cb = tf.keras.callbacks.ModelCheckpoint(filepath='./cp.weights.h5', save_weights_only=True, verbose=1)

history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=20,
  callbacks=[cb]
)

Epoch 1/20
374/375 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.7860 - loss: 0.7584
Epoch 1: saving model to ./cp.weights.h5
375/375 ━━━━━━━━━━━━━━━━━━━━ 13s 33ms/step - accuracy: 0.9054 - loss: 0.3542 - val_accuracy: 0.1138 - val_loss: 2.3040
Epoch 2/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.9709 - loss: 0.0988
Epoch 2: saving model to ./cp.weights.h5
375/375 ━━━━━━━━━━━━━━━━━━━━ 12s 32ms/step - accuracy: 0.9727 - loss: 0.0916 - val_accuracy: 0.9090 - val_loss: 0.2728
Epoch 3/20
374/375 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9798 - loss: 0.0680
Epoch 3: saving model to ./cp.weights.h5
375/375 ━━━━━━━━━━━━━━━━━━━━ 13s 33ms/step - accuracy: 0.9795 - loss: 0.0680 - val_accuracy: 0.8942 - val_loss: 0.3210
Epoch 4/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9832 - loss: 0.0544
Epoch 4: saving model to ./cp.weights.h5
375/375 ━━━━━━━━━━━━━━━━━━━━ 14s 36ms/step - accuracy: 0.9830 - loss: 0.0555 - val_accuracy: 0.9390 - val_loss: 0.2094
Epoch 5/20
3

In [14]:
test_loss, test_acc = model.evaluate(test_ds, verbose=2)
print('\nTest accuracy:', test_acc)

79/79 - 1s - 9ms/step - accuracy: 0.9673 - loss: 0.1419

Test accuracy: 0.9672999978065491


In [15]:
checkpoint_path = 'save/cnn-cp-{epoch:04d}.weights.h5'

In [16]:
model.save_weights(checkpoint_path.format(epoch=0))
print('Weights are saved successfully')

Weights are saved successfully
